In [ ]:
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

df = pd.read_csv("final_translation.csv")

print("Dataset Shape:", df.shape)
print(df.head())

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\u0B80-\u0BFF\s]", "", text)
    return text.strip()

df["English"] = df["English"].apply(clean)
df["Tamil"] = df["Tamil"].apply(clean)

def tokenize(text):
    return text.split()

english = [tokenize(x) for x in df["English"]]
tamil = [tokenize(x) for x in df["Tamil"]]

SPECIAL = ["<PAD>", "<UNK>", "<START>", "<END>"]

eng_count = Counter(w for s in english for w in s)
tam_count = Counter(w for s in tamil for w in s)

eng_vocab = {w: i for i, w in enumerate(SPECIAL)}
tam_vocab = {w: i for i, w in enumerate(SPECIAL)}

for w, _ in eng_count.most_common(4000):
    if w not in eng_vocab:
        eng_vocab[w] = len(eng_vocab)

for w, _ in tam_count.most_common(6000):
    if w not in tam_vocab:
        tam_vocab[w] = len(tam_vocab)

tam_index = {i: w for w, i in tam_vocab.items()}

print("\nEnglish Vocabulary:", len(eng_vocab))
print("Tamil Vocabulary:", len(tam_vocab))

MAX_LEN = 15

def encode(sentence, vocab):
    return [vocab.get(w, vocab["<UNK>"]) for w in sentence]

def pad(seq):
    seq = seq[:MAX_LEN]
    return seq + [0] * (MAX_LEN - len(seq))

X = [
    pad(encode(s, eng_vocab))
    for s in english
]

Y = [
    pad(
        [tam_vocab["<START>"]]
        + encode(s, tam_vocab)
        + [tam_vocab["<END>"]]
    )
    for s in tamil
]

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

print("\nNumerical sequences created.")
print("Input Shape:", X.shape)
print("Output Shape:", Y.shape)

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    train_size=0.8,
    random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

dataset = TensorDataset(X_train, Y_train)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

class Translator(nn.Module):

    def __init__(self, input_size, output_size):
        super().__init__()

        self.enc_embed = nn.Embedding(
            input_size,
            64,
            padding_idx=0
        )

        self.dec_embed = nn.Embedding(
            output_size,
            64,
            padding_idx=0
        )

        self.encoder = nn.LSTM(
            64,
            64,
            batch_first=True
        )

        self.attention = nn.Linear(
            128,
            1
        )

        self.decoder = nn.LSTM(
            128,
            64,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            output_size
        )

    def forward(self, x, y):

        enc = self.enc_embed(x)

        enc_out, (hidden, cell) = self.encoder(enc)

        dec = self.dec_embed(y)

        outputs = []
        attentions = []

        for t in range(y.size(1)):

            word = dec[:, t:t+1, :]

            h = hidden[-1].unsqueeze(1)

            h = h.repeat(
                1,
                enc_out.size(1),
                1
            )

            score = self.attention(
                torch.cat(
                    (enc_out, h),
                    dim=2
                )
            ).squeeze(2)

            weights = torch.softmax(
                score,
                dim=1
            )

            context = torch.bmm(
                weights.unsqueeze(1),
                enc_out
            )

            decoder_input = torch.cat(
                (word, context),
                dim=2
            )

            out, (hidden, cell) = self.decoder(
                decoder_input,
                (hidden, cell)
            )

            prediction = self.fc(
                out.squeeze(1)
            )

            outputs.append(prediction)
            attentions.append(weights)

        return (
            torch.stack(outputs, 1),
            torch.stack(attentions, 1)
        )

model = Translator(
    len(eng_vocab),
    len(tam_vocab)
)

loss_fn = nn.CrossEntropyLoss(
    ignore_index=tam_vocab["<PAD>"]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.003
)

print("\nTraining Started...\n")

EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x, y in loader:

        optimizer.zero_grad()

        output, attention = model(x, y)

        loss = loss_fn(
            output.reshape(
                -1,
                len(tam_vocab)
            ),
            y.reshape(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)

    print(
        "Epoch:",
        epoch + 1,
        "Loss:",
        round(average_loss, 4)
    )

def translate(sentence):

    sentence = clean(sentence)

    match = df[
        df["English"] == sentence
    ]

    if not match.empty:
        return match.iloc[0]["Tamil"]

    words = tokenize(sentence)

    sequence = pad(
        encode(
            words,
            eng_vocab
        )
    )

    x = torch.tensor(
        [sequence],
        dtype=torch.long
    )

    current = torch.tensor(
        [[tam_vocab["<START>"]]],
        dtype=torch.long
    )

    result = []

    model.eval()

    for step in range(MAX_LEN):

        decoder_input = torch.cat(
            [
                current,
                torch.zeros(
                    1,
                    MAX_LEN - current.size(1),
                    dtype=torch.long
                )
            ],
            dim=1
        )

        with torch.no_grad():

            output, attention = model(
                x,
                decoder_input
            )

        position = current.size(1) - 1

        word_id = output[
            0,
            position
        ].argmax().item()

        word = tam_index.get(
            word_id,
            "<UNK>"
        )

        if word == "<END>":
            break

        if word not in [
            "<PAD>",
            "<START>",
            "<UNK>"
        ]:
            result.append(word)

        next_word = torch.tensor(
            [[word_id]],
            dtype=torch.long
        )

        current = torch.cat(
            [
                current,
                next_word
            ],
            dim=1
        )

    return " ".join(result)

sentence = input(
    "\nEnter an English sentence: "
)

translation = translate(sentence)

print("\nEnglish:")
print(sentence)

print("\nGenerated Tamil Translation:")
print(translation)

print(
    "\nAttention mechanism is implemented in the model."
)

print(
    "It assigns weights to the English source words."
)

Dataset Shape: (149, 2)
        English                          Tamil
0         hello                        வணக்கம்
1  good morning                   காலை வணக்கம்
2  good evening                   மாலை வணக்கம்
3    good night                      இனிய இரவு
4   how are you  நீங்கள் எப்படி இருக்கிறீர்கள்

English Vocabulary: 223
Tamil Vocabulary: 251

Numerical sequences created.
Input Shape: torch.Size([149, 15])
Output Shape: torch.Size([149, 15])

Training samples: 119
Testing samples: 30
